# Train YCode-LM v5 (~385M parameters) on free Kaggle GPUs

This notebook grows **YCode-LM v4** (100M parameters, trained from scratch) into **YCode-LM v5**
(34 layers × 1024 wide, about 385M parameters). It then continues pretraining, instruction-tunes,
evaluates, and **publishes the model to your Hugging Face account**.

A Kaggle session lasts up to 12 hours, so training is split across sessions. Every ~90 minutes the
checkpoint is uploaded to a *private* Hugging Face repo. **Just run this notebook again** each time a
session ends; it resumes automatically and publishes the finished model at the end.

## One-time setup

1. **Settings → Accelerator → GPU T4 x2**.
2. **Settings → Internet → On**. Kaggle requires a verified phone number for this.
3. Create a Hugging Face token with **Write** access at https://huggingface.co/settings/tokens.
   In Kaggle, open **Add-ons → Secrets → Add a new secret**, name it `HF_TOKEN` and paste the token.
4. Set `HF_USER` below to your Hugging Face username.

## Each session

Click **Save Version → Save & Run All (Commit)** so it runs in the background for up to 12 hours,
even with your laptop closed. Check the log at the end to see the progress. When it says
**"All phases complete"**, your model is live at `https://huggingface.co/<HF_USER>/ycode-lm-v5`.

Expect about 3–5 sessions with the default settings. Kaggle's free quota is about 30 GPU-hours per week.

In [ ]:
# ---- Settings -------------------------------------------------------------
HF_USER = "Yazhiniyan"          # your Hugging Face username
MODEL_REPO = f"{HF_USER}/ycode-lm-v5"
PUBLIC = True                   # False keeps the finished model private
PRESET = "v5-385m"              # 34 x 1024, ~385M parameters
PRETRAIN_STEPS = 30000          # x 16k tokens/step (2 GPUs x batch 8 x 1024) ≈ 490M tokens
SFT_STEPS = 1000
SESSION_HOURS = 11.0            # stop starting new segments before Kaggle's 12 h limit
BRANCH = "claude/affectionate-rubin-7rl1am"   # switch to "main" once the pull request is merged

In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
if not os.path.isdir("Code-agent"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH,
                    "https://github.com/yazhiniyanMS/Code-agent.git"], check=True)
os.chdir("Code-agent")
subprocess.run(["pip", "install", "-q", "-e", ".[local,hub]"], check=True)

import torch
print("GPUs:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.is_available(), "Turn on a GPU accelerator in Settings."

In [ ]:
# Grows v4 -> v5 on the first run; on later runs it resumes from the Hugging Face checkpoint.
!python -m ycode.lm.cli autotrain \
    --base-model models/ycode-lm-v4 --preset {PRESET} \
    --hf-repo {MODEL_REPO} {"--public" if PUBLIC else ""} \
    --pretrain-steps {PRETRAIN_STEPS} --sft-steps {SFT_STEPS} \
    --batch-size 8 --hours {SESSION_HOURS} --segment-minutes 90 --max-mb 1000

## After it finishes

Try the model here, or locally after downloading it from Hugging Face:

```bash
huggingface-cli download <HF_USER>/ycode-lm-v5 --local-dir models/ycode-lm-v5
ycode --local models/ycode-lm-v5
```

In [ ]:
import os
if os.path.exists("ycode-autotrain/release/model.pt"):
    from ycode.lm.generate import LocalLM
    lm = LocalLM("ycode-autotrain/release")
    print(lm.chat([("user", "Write a Python function `is_even(n)` that returns True if n is even.")],
                  max_new_tokens=150, temperature=0.2))
else:
    print("Not finished yet: run the notebook again in a new session to continue training.")
